In [ ]:
import os
import numpy as np

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
#os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
import torch
#torch.set_num_threads(1)
from cellpose import models, io, train
io.logger_setup()
# 1. Define your folder paths
train_dir = '/Users/nadiaayad/Documents/Models/T-mNeonGreen/Training'
test_dir = '/Users/nadiaayad/Documents/Models/T-mNeonGreen/Testing'

# 2. Load the images and masks automatically
# The mask_filter argument ensures it looks for files ending in '_masks.tif' or '_masks.png'
print("Loading data...")
output = io.load_train_test_data(train_dir, test_dir, mask_filter='_seg.npy', look_one_level_down = False)
train_data, train_labels, train_image_names, test_data, test_labels, test_image_names = output
print(f"Successfully loaded {len(train_data)} training images!")

for i in range(len(train_data)):
    if train_data[i].ndim>2:
        train_data[i] = np.squeeze(train_data[i])

    train_labels[i] = (train_labels[i]>0).astype(np.uint16)

print(f"Cleaned Image Shape: {train_data[0].shape}")
print(f"Cleaned Mask Unique Values: {np.unique(train_labels[0])}")

# --- APPLE SILICON (M3) GPU SETUP ---
if torch.backends.mps.is_available():
    m3_device = torch.device("mps")
    use_gpu = True
    print("Apple M3 GPU (MPS) detected and activated!")
else:
    m3_device = torch.device("cpu")
    use_gpu = False
    print("MPS not found. Defaulting to CPU.")

# 3. Initialize the base model with the M3 device
print("Initializing model...")
model = models.CellposeModel(
    gpu=use_gpu, 
    device=m3_device, # Forces Cellpose to use the Mac GPU
    pretrained_model='nuclei'
)

# 4. Train the model
print("Starting fine-tuning...")
new_model_path, train_losses, test_losses = train.train_seg(
    model.net,             # You now pass the underlying PyTorch network
    train_data=train_data, 
    train_labels=train_labels, 
    test_data=test_data, 
    test_labels=test_labels,
    save_path=train_dir,   # Where to save the new model
    n_epochs=100,          # Lower epochs for small datasets
    learning_rate=0.00001, 
    weight_decay=0.1,
    batch_size=5,
    min_train_masks=1,
    model_name='t-reporter_model' # Your custom name
)

print(f"\nTraining complete! Your new model is saved at:\n{new_model_path}")

2026-06-24 14:54:03,686 [INFO] WRITING LOG OUTPUT TO /Users/nadiaayad/.cellpose/run.log
2026-06-24 14:54:03,687 [INFO] 
cellpose version: 	4.1.1 
platform:       	darwin 
python version: 	3.10.20 
torch version:  	2.10.0
Loading data...
2026-06-24 14:54:03,692 [INFO] not all flows are present, running flow generation for all images
2026-06-24 14:54:03,853 [INFO] 10 / 10 images in /Users/nadiaayad/Documents/Models/T-mNeonGreen/Training folder have labels
2026-06-24 14:54:03,863 [INFO] not all flows are present, running flow generation for all images
2026-06-24 14:54:03,877 [INFO] 2 / 2 images in /Users/nadiaayad/Documents/Models/T-mNeonGreen/Testing folder have labels
Successfully loaded 10 training images!
Cleaned Image Shape: (297, 297)
Cleaned Mask Unique Values: [0 1]
Apple M3 GPU (MPS) detected and activated!
Initializing model...
2026-06-24 14:54:03,883 [WARNING] pretrained model /Users/nadiaayad/.cellpose/models/cpsam not found, using default model
